# {Project Title}📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
📝 <!-- Answer Below -->

Historically, women have been underrepresented when it comes to health research and information. Female and male bodies are biologically different and can behave differently under the same circumstances or conditions, so it is very important that adequate research is done on both sexes so both halves of the population are able to be diagnosed and cared for correctly. There have been circumstances where female patients who are actively experiencing heart attacks are misdiagnosed or sent home because the common symptoms associated with it were drawn from research on men while women show different warning signs. 

I want to investigate the ratio of male and female representation in clincial trials overtime and see how it matches with the ratio of each sex experiencing that condition (their burden ratio). 

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
📝 <!-- Answer Below -->

Is there a gap in sex representation in clinical trials compared to their reported burden of the condition? Has this changed overtime?

Are there specific conditions that are more overlooked than others?

Is there a correlation between better representation and higher funding for different conditions?

## What would an answer look like?
*What is your hypothesized answer to your question?*
📝 <!-- Answer Below -->

The answer will be the difference between the ratio of male and female representation in trials and the ratio of male and female burden of that condition. The difference can be calculated initially by condition, then by year, and then all together to show how that has evolved, if at all, overtime. 

Each step in calculations will answer different parts of our overarching question. The first step will reveal if there are specific conditions that are more overlooked than others, then how much of a gap there is per year, and finally if a gap exists, how it has changed through the years. 

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
📝 <!-- Answer Below -->

#### AACT 
*Aggregate Analysis of ClinicalTrials.gov*

Extensive data collection on every clinical study registered in ClinicalTrials.gov. The data includes necessary information for the project question like trial dates, participants, sex of participants, and condition being studied. It also includes a variety of extra information that can be analyzed for this project such as trial phase, involvement of FDA regulated drugs/devices, and accepted participant age limits.

#### GBD
*Global Burden of Disease*

Database of medical conditions and how much they are experienced globally (2008-2022). The data can be filtered by age, location, date, cause, measure, and more. For this project it will return the number of people reported to have experienced certain conditions globally by sex and year. The measure for people experiencing a condition will be "Prevalence", which reports how many people are experiencing it at a given time (a specific year). 

We will also use the "DALY" (Disability-Adjusted Life Years) measure to rank and choose the conditions that we will be comparing trial ratios on. DALY sums the years of life lost due to premature death with years lived with disability to count the years of healthy years lost. This way we can find the most prevalent conditions that are also more serious and more likely to require clinical research and trials for.

#### NIH RCDC
*National Institute of Health, Research, Condition, and Disease*

Reports the annual monetary support level for conditions measured by grants, contracts, and other funding mechanisms used at the National Institute of Health (NIH). This will allow for analysis to see if there is a correlation with conditions that receive more funding and their levels of representation in clinical trials.

### Connecting the Data

These datasets need to be connected on the basis on condition, but each define conditions with different terms or at different levels of generality. So to connect them we need to make hand-verified crosswalk tables that are keyed on condition and year. The top 15 to 20 Level 3 GBD causes by 2022 DALYs will be picked after removing all sex-specific, "other", and unmappable causes. GBD cause will match on AACT MeSH terms. Each cause will also be mapped to its closest NIH RCDC category, which will be joined on category and fiscal year. 

## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*
📝 <!-- Start Discussing the project here; you can add as many code cells as you need -->

Using the data, I will be able to compare the participation in clinical trials by sex compared to the condition burden each sex reports. This will reveal if there are gaps between those who experience different conditions, and who is actually included in research for that. With data spanning from 2008-2022, we will be able to compare these gaps overtime to see if they have widened, stayed the same, or improved. 

Using the male and female share of prevalence from the GBD by condition, I can compare this ratio with the male and female share of enrolled participants from trials that report sex. The difference between these two ratios is the participation gap, which I will chart by condition and track overtime. Using the NIH RCDC set, I can join by condition and year to explore any possible correlations between wider participation gaps and amount of funding. This portion is limited, however, by RCDC only covering US funding while GBD is global.

### AACT
Connected by establishing connection to database with credentials generated after creating account. ([https://aact.ctti-clinicaltrials.org/](https://aact.ctti-clinicaltrials.org/)).

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text
import pandas as pd

load_dotenv()

# -- AACT
# Establish connection
url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("AACT_USER"),
    password=os.getenv("AACT_PASSWORD"),
    host="aact-db.ctti-clinicaltrials.org",
    port=5432,
    database="aact",
)
engine = create_engine(url)

with engine.connect() as conn:
    df = pd.read_sql(text("SELECT nct_id, brief_title FROM studies LIMIT 5"), conn)
df

,nct_id,brief_title
0,NCT05917496,Analysis of Parental Support in Families Using...
1,NCT05945641,Effect of Low-load Resistance Training vs. Hig...
2,NCT02688673,DC Vaccine Combined With CIK Cells in Patients...
3,NCT00332839,Comparison of CNI-based Regimen Versus CNI-fre...
4,NCT00638560,Changes of Biomarkers in Response to Training ...


### GBD
Located in `data/` folder, downloaded from filterable queries and download abilities directly from site ([vizhub.healthdata.org/gbd-results](vizhub.healthdata.org/gbd-results)).

**File Name:** GBD_2008_2023.csv

#### Filters
| Filter | Selection |
|---|---|
| GBD Estimate | Cause of death or injury |
| Measure | Deaths, DALYs, Prevalence, Incidence |
| Metric | Number |
| Cause | All level 3 |
| Location | Global |
| Age | All ages |
| Sex | Male, Female (not Both, each selected separately) |
| Total percentage change | False |
| Year | 2008–2022 |

### NIH RCDC
Downloaded directly from [https://report.nih.gov/funding/categorical-spending#/](https://report.nih.gov/funding/categorical-spending#/) with "Download as Excel". Converted to CSV through Excel, cleaned extra rows above and below data, and uploaded as "/data/RCDC_2008_2025".

## Data Cleaning and Transformation

### GBD

In [16]:
import os
import pandas as pd

os.makedirs("data/processed", exist_ok=True)

gbd = pd.read_csv("data/GBD_2008_2023.csv")
gbd.info()

<class 'pandas.DataFrame'>
RangeIndex: 16416 entries, 0 to 16415
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   population_group_id    16416 non-null  int64  
 1   population_group_name  16416 non-null  str    
 2   measure_id             16416 non-null  int64  
 3   measure_name           16416 non-null  str    
 4   location_id            16416 non-null  int64  
 5   location_name          16416 non-null  str    
 6   sex_id                 16416 non-null  int64  
 7   sex_name               16416 non-null  str    
 8   age_id                 16416 non-null  int64  
 9   age_name               16416 non-null  str    
 10  cause_id               16416 non-null  int64  
 11  cause_name             16416 non-null  str    
 12  metric_id              16416 non-null  int64  
 13  metric_name            16416 non-null  str    
 14  year                   16416 non-null  int64  
 15  val          

Dropping 2023 so the years match the 2008–2022 window, and keeping only the columns needed. Location, age, and metric only have one value each, so they are removed.

In [17]:
gbd = gbd[gbd["year"] <= 2022]
gbd = gbd[["measure_name", "sex_name", "cause_name", "year", "val"]]
gbd["measure_name"] = gbd["measure_name"].replace("DALYs (Disability-Adjusted Life Years)", "DALYs")

print(gbd.isna().sum())
print("Duplicates:", gbd.duplicated().sum())

measure_name    0
sex_name        0
cause_name      0
year            0
val             0
dtype: int64
Duplicates: 0


Male and Female are moved into their own columns so the female share can be calculated. Sex-specific causes like prostate cancer only have rows for one sex, so the missing sex is filled with 0. Values are decimals because GBD numbers are model estimates, not exact counts.

In [18]:
gbd = gbd.pivot_table(index=["measure_name", "cause_name", "year"], columns="sex_name", values="val").reset_index()
gbd = gbd.fillna(0)
gbd["Both"] = gbd["Male"] + gbd["Female"]
gbd["female_share"] = gbd["Female"] / gbd["Both"]
gbd.head()

sex_name,measure_name,cause_name,year,Female,Male,Both,female_share
0,DALYs,Acne vulgaris,2008,488314.264982,381146.293384,869460.558366,0.561629
1,DALYs,Acne vulgaris,2009,497795.236594,387273.275972,885068.512566,0.562437
2,DALYs,Acne vulgaris,2010,505425.887925,392476.443138,897902.331063,0.562896
3,DALYs,Acne vulgaris,2011,512635.364568,398018.903071,910654.267639,0.562931
4,DALYs,Acne vulgaris,2012,520291.066093,403806.812161,924097.878254,0.563026


#### Choosing conditions
Ranking causes by 2022 DALYs and dropping:
- Sex-specific causes (one sex has less than 5% of prevalence)
- Causes with no prevalence data
- "Other" catch-all categories
- Causes that don't have data for every year

In [19]:
dalys_2022 = gbd[(gbd["measure_name"] == "DALYs") & (gbd["year"] == 2022)]
prev_2022 = gbd[(gbd["measure_name"] == "Prevalence") & (gbd["year"] == 2022)]

causes = dalys_2022[["cause_name", "Both"]].rename(columns={"Both": "dalys_2022"})
causes = causes.merge(prev_2022[["cause_name", "female_share"]], on="cause_name", how="left")
causes = causes.sort_values("dalys_2022", ascending=False)

zero_dalys = gbd[(gbd["measure_name"] == "DALYs") & (gbd["Both"] == 0)]["cause_name"].unique()

sex_specific = (causes["female_share"] < 0.05) | (causes["female_share"] > 0.95)
no_prevalence = causes["female_share"].isna()
other = causes["cause_name"].str.startswith("Other") | (causes["cause_name"] == "Endocrine, metabolic, blood, and immune disorders")
not_all_years = causes["cause_name"].isin(zero_dalys)

drop = sex_specific | no_prevalence | other | not_all_years
dropped_causes = causes[drop]
causes = causes[~drop]
dropped_causes

sex_name,cause_name,dalys_2022,female_share
21,COVID-19,7.691844e+07,0.518460
55,Gynecological diseases,2.713903e+07,1.000000
20,Breast cancer,2.369258e+07,0.983028
46,"Endocrine, metabolic, blood, and immune disorders",1.423292e+07,0.660289
23,Cervical cancer,1.275439e+07,1.000000
134,Urinary diseases and male infertility,1.177449e+07,0.043034
99,Other cardiovascular and circulatory diseases,1.115996e+07,0.469973
109,Prostate cancer,8.671374e+06,0.000000
102,Ovarian cancer,6.240822e+06,1.000000
9,Aortic aneurysm,3.300405e+06,NaN


The top 15 causes are used, after removing four that are poor fits:
- **Congenital birth defects:** too broad to match to trial terms or funding categories
- **Dietary iron deficiency:** many iron trials are in pregnant women, which skews the female share
- **Low back pain:** no RCDC funding data before 2016
- **Age-related and other hearing loss:** few trials with results, mostly device studies

In [20]:
weak_causes = ["Congenital birth defects", "Dietary iron deficiency", "Low back pain", "Age-related and other hearing loss"]
top_causes = causes[~causes["cause_name"].isin(weak_causes)].head(15)
top_causes

sex_name,cause_name,dalys_2022,female_share
69,Ischemic heart disease,1.893905e+08,0.425472
123,Stroke,1.564841e+08,0.462266
79,Lower respiratory infections,9.278106e+07,0.488978
37,Diabetes mellitus,8.820050e+07,0.466009
26,Chronic obstructive pulmonary disease,7.144156e+07,0.512268
38,Diarrheal diseases,5.965194e+07,0.539702
35,Depressive disorders,5.447478e+07,0.607204
81,Malaria,5.357163e+07,0.505300
8,Anxiety disorders,4.788576e+07,0.615741
128,"Tracheal, bronchus, and lung cancer",4.654881e+07,0.351412


In [21]:
gbd = gbd[gbd["cause_name"].isin(causes["cause_name"])]

gbd.to_csv("data/processed/gbd_clean.csv", index=False)
top_causes.to_csv("data/processed/gbd_top_causes.csv", index=False)
dropped_causes.to_csv("data/processed/gbd_dropped_causes.csv", index=False)

### NIH RCDC

In [24]:
rcdc = pd.read_csv("data/RCDC_2008_2025.csv")
rcdc = rcdc.rename(columns={rcdc.columns[0]: "category"})
rcdc.head()

,category,2008,2009,2009 ARRA,2010,2010 ARRA,2011,2012,2013,2014,...,2018,2019,2020,2021,2022,2023,2024,2025,2024 US Mortality 14,2024 US Prevalence SE 14
0,Acquired Cognitive Impairment,+,+,+,+,+,+,+,+,+,...,"$1,978","$2,423","$2,897","$3,259","$3,513","$3,691","$3,768","$3,789",-,-
1,Acute Respiratory Distress Syndrome,$82,$103,$17,$110,$22,$96,$98,$95,$85,...,$123,$126,$158,$148,$140,$156,$151,$160,"12,386",-
2,Adolescent Sexual Activity,+,+,+,$80,$7,$69,$76,$70,$68,...,$96,$102,$88,$94,$81,$94,$73,$62,-,-
3,Adult/Somatic Stem Cell and Progenitor Cell Re...,+,+,+,+,+,+,+,+,+,...,+,+,+,+,+,+,+,"$1,393",-,-
4,Agent Orange & Dioxin,$13,$13,$2,$11,$1,$8,$8,$10,$8,...,$10,$8,$6,$8,$9,$6,$5,$6,-,-


Keeping only the 2008–2022 year columns. The ARRA columns are dropped because they were one-time stimulus funding, and the 2024 mortality and prevalence columns aren't funding. Footnote numbers at the end of category names (e.g. "Diabetes 5") are removed.

In [25]:
years = [str(y) for y in range(2008, 2023)]
rcdc = rcdc[["category"] + years]
rcdc["category"] = rcdc["category"].str.replace(r"[\d,\s]+$", "", regex=True)
print("Duplicate categories:", rcdc["category"].duplicated().sum())

Duplicate categories: 0


Funding values are text like `1,132` with a dollar sign in front, so the dollar signs and commas are removed to make them numbers. Cells with `+` or `*` aren't dollar amounts, so they become missing (NaN) instead of 0.

In [26]:
for year in years:
    rcdc[year] = pd.to_numeric(rcdc[year].str.replace("$", "").str.replace(",", ""), errors="coerce")

print(rcdc.isna().sum())

category      0
2008        137
2009        133
2010        122
2011        118
2012        116
2013        116
2014        109
2015         89
2016         72
2017         69
2018         67
2019         61
2020         52
2021         42
2022         35
dtype: int64


Reshaping so each row is one category and year, which makes it easier to join with GBD later.

In [27]:
rcdc = rcdc.melt(id_vars="category", var_name="year", value_name="funding_millions")
rcdc["year"] = rcdc["year"].astype(int)

rcdc.to_csv("data/processed/rcdc_clean.csv", index=False)
rcdc.head()

,category,year,funding_millions
0,Acquired Cognitive Impairment,2008,NaN
1,Acute Respiratory Distress Syndrome,2008,82.0
2,Adolescent Sexual Activity,2008,NaN
3,Adult/Somatic Stem Cell and Progenitor Cell Re...,2008,NaN
4,Agent Orange & Dioxin,2008,13.0


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

In [9]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

[NbConvertApp] Converting notebook source.ipynb to python
[NbConvertApp] Writing 6593 bytes to source.py
